# Widgets 🎛️

The widgets of [GoNB](https://github.com/janpfeifer/gonb), the Go kernel for
Jupyter (buttons, sliders and selects), work in gopyter with GoNB's own API,
drawn in the terminal. While a cell runs:

- press `enter` on it to focus the first widget, and `tab` to move to the next;
- `←`/`→` move a slider (`pgup`/`pgdown` by 10%) or change a select, `enter`
  presses a button or lists a select's options;
- or use the mouse: click a slider's track, a button or a select;
- press **✓ done** (or `ctrl+d`) when you're finished: every `Listen` channel
  closes, so the loops below end. `ctrl+c` interrupts instead.

With `gopyter run`, nobody can use the widgets, so they are done at once.

In [ ]:
import (
	"image"
	"image/color"
	"math"

	"github.com/janpfeifer/gonb/gonbui/dom"
	"github.com/janpfeifer/gonb/gonbui/widgets"
)

var paletteNames = []string{"ocean", "fire", "mono"}

// Mandelbrot renders the set around (cx, cy); scale is the width of the view.
func Mandelbrot(w, h int, cx, cy, scale float64, maxIter, palette int) image.Image {
	img := image.NewRGBA(image.Rect(0, 0, w, h))
	for py := range h {
		for px := range w {
			x := cx + (float64(px)/float64(w)-0.5)*scale
			y := cy + (float64(py)/float64(h)-0.5)*scale*float64(h)/float64(w)
			img.Set(px, py, shade(x, y, maxIter, palette))
		}
	}
	return img
}

func shade(x, y float64, maxIter, palette int) color.Color {
	var zx, zy float64
	for n := range maxIter {
		zx, zy = zx*zx-zy*zy+x, 2*zx*zy+y
		if r2 := zx*zx + zy*zy; r2 > 16 {
			t := (float64(n) + 1 - math.Log2(math.Log(r2)/2)) / 40
			c := func(phase float64) uint8 { return uint8(127.5 + 127.5*math.Cos(2*math.Pi*(t+phase))) }
			switch palette {
			case 1:
				return color.RGBA{c(0), c(0.15), c(0.35), 255}
			case 2:
				return color.Gray{c(0)}
			}
			return color.RGBA{c(0.5), c(0.3), c(0.1), 255}
		}
	}
	return color.Black
}

## Explore the Mandelbrot set

Zoom in with the first slider, raise the detail as you go, and try the
palettes. The `%%` line, as in GoNB, runs the rest as `main()`: its variables
stay in this cell.

In [ ]:
%%
div := dom.CreateTransientDiv()
dom.Append(div, "<b>zoom</b> ")
zoom := widgets.Slider(0, 120, 0).AppendTo(div).Done()
dom.Append(div, " <b>detail</b> ")
detail := widgets.Slider(50, 1500, 150).AppendTo(div).Done()
dom.Append(div, " ")
palette := widgets.Select(paletteNames).AppendTo(div).Done()

// The values arrive on the channels; keep the latest of each.
z, d, p := zoom.Value(), detail.Value(), palette.Value()
draw := func() {
	scale := 3.0 * math.Pow(0.9, float64(z))
	nb.DisplayID("fractal", Mandelbrot(240, 150, -0.7436447860, 0.1318252536, scale, d, p))
}
draw()

zc, dc, pc := zoom.Listen().LatestOnly(), detail.Listen().LatestOnly(), palette.Listen()
loop:
for {
	// A closed channel means the user pressed done.
	select {
	case v, ok := <-zc.C:
		if !ok {
			break loop
		}
		z = v
	case v, ok := <-dc.C:
		if !ok {
			break loop
		}
		d = v
	case v, ok := <-pc.C:
		if !ok {
			break loop
		}
		p = v
	}
	draw()
}
fmt.Printf("final view: zoom %d, detail %d, %s palette\n", z, d, paletteNames[p])

## Buttons and live HTML

In [ ]:
%%
// dom changes displayed HTML by element id: here, the counter.
div := dom.CreateTransientDiv()
dom.Append(div, `Clicked <b id="clicks">0</b> times `)
plus := widgets.Button("+1").AppendTo(div).Done()
reset := widgets.Button("reset").AppendTo(div).Done()

clicks := 0
pc, rc := plus.Listen(), reset.Listen()
for {
	select {
	case _, ok := <-pc.C:
		if !ok {
			return
		}
		clicks++
	case _, ok := <-rc.C:
		if !ok {
			return
		}
		clicks = 0
	}
	dom.SetInnerText("clicks", strconv.Itoa(clicks))
}

## Sliders with markdown

In [ ]:
%%
feel := func(c int) string {
	switch {
	case c < 0:
		return "🥶 freezing"
	case c < 15:
		return "🧥 chilly"
	case c < 25:
		return "😎 pleasant"
	case c < 35:
		return "🥵 hot"
	}
	return "🔥 scorching"
}
show := func(c int) {
	nb.DisplayMarkdownID("temp", fmt.Sprintf("**%d °C** is **%.1f °F**: %s", c, float64(c)*9/5+32, feel(c)))
}

celsius := widgets.Slider(-30, 50, 20).Done()
show(celsius.Value())
for c := range celsius.Listen().LatestOnly().C {
	show(c)
}